# Tesla Stock Opening Price Prediction using ARIMA and LSTM
## Comparative Analysis of Statistical and Deep Learning Models

## 1. Import Required Libraries

In [ ]:
!pip install tensorflow

In [ ]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error,r2_score
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf

## 2. Data Acquisition and Loading

In [ ]:
df=pd.read_csv(r'C:\Users\ashld\Downloads\archive\TSLA.csv')

In [ ]:
df.head()

## 3. Data Preprocessing

In [ ]:
df['Date']=pd.to_datetime(df['Date'])

In [ ]:
df.set_index('Date',inplace=True)

In [ ]:
df.head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(df['Open'])
plt.title("Tesla Stock opening price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

In [ ]:
print(df.isnull().sum())

## 5. Stationarity Analysis(ADF Test)

In [ ]:
result=adfuller(df['Open'])
print("ADF Statistics:",result[0])
print("p-value:",result[1])

### 5.1 First Order Differencing

In [ ]:
df_diff=df['Open'].diff().dropna()

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(df_diff)
plt.title("Differenced Tesla Stock opening price")
plt.show()

In [ ]:
result=adfuller(df_diff)
print("ADF Statistics:",result[0])
print("p-value:",result[1])

## 6. ACF and PACF analysis

In [ ]:
plt.figure(figsize=(10,6))
plot_acf(df_diff,lags=40)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plot_pacf(df_diff,lags=40)
plt.show()

In [ ]:
best_aic=float('inf')
best_order=None
for p in range(6):
    for d in range(2):
        for q in range(6):
            try:
                model=ARIMA(df['Open'],order=(p,d,q))
                result=model.fit()
                if result.aic < best_aic:
                    best_aic=result.aic
                    best_order=(p,d,q)
            except:
                continue
print("Best Order:",best_order)
print("Best AIC:",best_aic)                    

## 7. ARIMA Modeling and Forecasting

In [ ]:
train_size=int(len(df)*0.8)
train=df['Open'][:train_size]
test=df['Open'][train_size:]
print(train.shape)
print(test.shape)

In [ ]:
model=ARIMA(train,order=(5,1,4))
result=model.fit()

In [ ]:
forecast=result.forecast(steps=len(test))

In [ ]:
forecast.size

In [ ]:
rmse=np.sqrt(mean_squared_error(test,forecast))
r2=r2_score(test,forecast)
print("RMSE:",rmse)
print("R2:",r2)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(test.index,test,label="Actual")
plt.plot(test.index,forecast,label="ARIMA Forecast")
plt.title("Tesla Opening Price Prediction using ARIMA")
plt.xlabel("Time")
plt.ylabel("Opening Price")
plt.legend()
plt.show()

## 8. Data Preparation for LSTM

In [ ]:
data=df[['Open']]

### 8.1 Min-Max Scaling

In [ ]:
scaler=MinMaxScaler(feature_range=(0,1))
data=scaler.fit_transform(data)

In [ ]:
print(data[:5])
print(data.shape)

### 8.2 Creating 60-day sequencem

In [ ]:
X=[]
y=[]
window_size=60
for i in range(window_size,len(data)):
    X.append(data[i-window_size:i,0])
    y.append(data[i,0])
X=np.array(X)
y=np.array(y)

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
X=X.reshape(X.shape[0],X.shape[1],1)
print(X.shape)

### 8.3 Train test split

In [ ]:
train_size=int(len(X)*0.8)
X_train=X[:train_size]
X_test=X[train_size:]
y_train=y[:train_size]
y_test=y[train_size:]

## 9. LSTM Architecture

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

model=Sequential()
model.add(
    LSTM(
        units=50,
        return_sequences=True,
        input_shape=(60,1)
    )
)
model.add(Dropout(0.2))
model.add(
    LSTM(
        units=50,
        return_sequences=False
    )
)
model.add(Dropout(0.2))
model.add(Dense(25))
model.add(Dense(1))

In [ ]:
model.compile(optimizer='adam',loss='mean_squared_error')

In [ ]:
print(tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

In [ ]:
model.summary()

## 10. Model Training

In [ ]:
history=model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test,y_test),
    verbose=1
    )

In [ ]:
print(history.history.keys())

In [ ]:
print("Final Training Loss:",history.history['loss'][-1])
print("Final Validation Loss:",history.history['val_loss'][-1])

## 11. LSTM Predictions and Evaluation

In [ ]:
predictions=model.predict(X_test)
predictions=scaler.inverse_transform(predictions)
y_test_actual=scaler.inverse_transform(y_test.reshape(-1,1))

In [ ]:
rmse=np.sqrt(mean_squared_error(y_test_actual,predictions))
r2=r2_score(y_test_actual,predictions)
print("RMSE:",rmse)
print("R2:",r2)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(y_test_actual,label="Actual")
plt.plot(predictions,label="Predicted")
plt.title("Tesla stock Opening Price Predction using LSTM")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Training vs Validation Loss')
plt.show()